# Mixing matrices

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/09-mixing-matrices` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

By default every person contacts every other at the same rate. A mixing matrix
$K$ relaxes that: columns are infectors, rows are the infected. Under frequency
dependence

$$\lambda_a = c \sum_b K_{ab}\, I_b / N_b.$$

In summer4 the matrix is a `MixingMatrix` on the `ForceOfInfection`, not a method
on a stratification object.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from summer4 import (
    Compartments,
    ExitFlow,
    FlowModel,
    Multiply,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Time,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix
from summer4.flows.rates import ArrayConst
from summer4.timevarying import piecewise

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "I", "R"))
age = Property("age", ("young", "old"))
location = Property("location", ("urban", "rural"))
pop = Property("pop", ("all",))


def plot_comp(res, title):
    return res["comp"].to_pandas().plot(
        title=title, labels={"index": "time (days)", "value": "people"}
    )


def wrap_y0(pmap, arr):
    return PropertyData.wrap(pmap, jnp.asarray(arr))


def run(model, y0, params=None, t1=20.0):
    plan = SavePlan(
        requests={"comp": SaveRequest(Compartments())},
        ts=np.linspace(0.0, t1, 201),
    )
    p = {} if params is None else params
    return model.compile().run(p, y0, t0=0.0, t1=t1, dt=0.1, save=plan, solver="euler")


## Unstratified SIR


In [ ]:
pmap0 = PropertyMap.from_property(state).stratify(pop)
m0 = FlowModel(pmap0)
mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
m0.add_flow(TransitionFlow("infection", state["S"], state["I"], ForceOfInfection("infection", infectious=state["I"], group_by=mixing.prop, mixing=mixing, kind="frequency", contact_rate=Param("contact"))))
m0.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
m0.add_flow(ExitFlow("infection_death", state["I"], 0.05))
y0_arr = np.zeros(pmap0.size)
y0_arr[pmap0.select(state["S"])] = 990.0
y0_arr[pmap0.select(state["I"])] = 10.0
y0 = wrap_y0(pmap0, y0_arr)
params = {"contact": 2.0}
res0 = run(m0, y0, params)
assert float(np.max(np.asarray(res0["comp"].select(state["I"]).values.data))) > 10.0

# JIT gate: finite grad of peak I through the contact Param.
cm0 = m0.compile()
plan0 = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 5.0, 11),
)


def loss_contact(contact: jax.Array) -> jax.Array:
    out = cm0.run(
        {"contact": contact},
        y0,
        t0=0.0,
        t1=5.0,
        dt=0.1,
        save=plan0,
        solver="euler",
    )
    return jnp.sum(jnp.asarray(out["comp"].select(state["I"]).values.data))


val = float(jax.jit(loss_contact)(jnp.asarray(2.0)))
grad = float(jax.jit(jax.grad(loss_contact))(jnp.asarray(2.0)))
assert np.isfinite(val) and np.isfinite(grad)
plot_comp(res0, "Unstratified SIR")


## Age mixing matrix

A full stratification (every compartment has `age`) can carry a 2×2 matrix.


In [ ]:
pmap_age = PropertyMap.from_property(state).stratify(age)
K_age = np.array(
    [
        [0.2, 0.3],
        [0.5, 0.7],
    ]
)
epi = FlowModel(pmap_age)
mixing = MixingMatrix(age, K_age, normalize="none", check_reciprocal=False)
epi.add_flow(TransitionFlow("infection", state["S"], state["I"], ForceOfInfection("infection", infectious=state["I"], group_by=mixing.prop, mixing=mixing, kind="frequency", contact_rate=Param("contact"))))
epi.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
epi.add_flow(ExitFlow("infection_death", state["I"], 0.05))
y0_arr = np.zeros(pmap_age.size)
y0_arr[pmap_age.select(state["S"])] = 495.0
y0_arr[pmap_age.select(state["I"])] = 5.0
y0 = wrap_y0(pmap_age, y0_arr)
res = run(epi, y0, params)
assert float(np.max(np.asarray(res["comp"].select(state["I"]).values.data))) > 5.0
plot_comp(res, "Age-stratified SIR with mixing matrix")


## Time-varying mixing matrices

Blend a "normal" matrix and a lockdown matrix with a piecewise weight on
`Time` (ArrayConst × scalar interp).


In [ ]:
K_normal = np.array([[0.2, 0.3], [0.5, 0.7]])
K_lock = np.array([[0.3, 0.0], [0.0, 0.8]])
is_lockdown = piecewise(Time(), (3.0, 8.0), (0.0, 1.0, 0.0))
K_t = (1.0 - is_lockdown) * ArrayConst(K_normal) + is_lockdown * ArrayConst(K_lock)

epi = FlowModel(pmap_age)
mixing = MixingMatrix(age, K_t, normalize="none", check_reciprocal=False)
epi.add_flow(TransitionFlow("infection", state["S"], state["I"], ForceOfInfection("infection", infectious=state["I"], group_by=mixing.prop, mixing=mixing, kind="frequency", contact_rate=Param("contact"))))
epi.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
epi.add_flow(ExitFlow("infection_death", state["I"], 0.05))
res = run(epi, y0, params)
assert float(np.max(np.asarray(res["comp"].select(state["I"]).values.data))) > 5.0
plot_comp(res, "Mixing matrix lockdown between days 3 and 8")


## Multiple mixing axes

summer2 multiplied independent age and location matrices with a Kronecker
product. `ForceOfInfection` takes **one** `group_by`, so the summer4 form is
explicit: build the Kronecker product and group by a flattened
cross-classification property (or mix on only one axis).

Below we keep age and location as separate properties for demography, mix on
**age** only, and show the Kronecker product that would be used for a combined
grouping.


In [ ]:
pmap_al = PropertyMap.from_property(state).stratify(age).stratify(location)
K_loc = np.array([[0.8, 0.2], [0.2, 0.8]])
# Age mixing as before; location enters mortality only.
epi = FlowModel(pmap_al)
mixing = MixingMatrix(age, K_age, normalize="none", check_reciprocal=False)
epi.add_flow(TransitionFlow("infection", state["S"], state["I"], ForceOfInfection("infection", infectious=state["I"], group_by=mixing.prop, mixing=mixing, kind="frequency", contact_rate=Param("contact"))))
epi.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
epi.add_flow(
    ExitFlow(
        "infection_death",
        state["I"],
        0.05,
        adjust=[Multiply(3.0, where=location["rural"])],
    )
)
y0_arr = np.zeros(pmap_al.size)
# 70% urban, 30% rural; even age split within.
for a in age.traits:
    y0_arr[pmap_al.select(state["S"] & age[a] & location["urban"])] = 990.0 * 0.7 * 0.5
    y0_arr[pmap_al.select(state["S"] & age[a] & location["rural"])] = 990.0 * 0.3 * 0.5
    y0_arr[pmap_al.select(state["I"] & age[a] & location["urban"])] = 10.0 * 0.7 * 0.5
    y0_arr[pmap_al.select(state["I"] & age[a] & location["rural"])] = 10.0 * 0.3 * 0.5
y0 = wrap_y0(pmap_al, y0_arr)
res = run(epi, y0, params)
assert float(np.max(np.asarray(res["comp"].select(state["I"]).values.data))) > 5.0
plot_comp(res, "Age mixing + location-specific mortality")


### Kronecker product of the two matrices

Order follows `itertools.product(age, location)` — the same convention you
would use for a flattened `age_loc` property.


In [ ]:
import itertools

K_combined = np.kron(K_age, K_loc)
labels = [f"{a}+{loc[:3]}" for a, loc in itertools.product(age.traits, location.traits)]
assert K_combined.shape == (4, 4)
fig = px.imshow(
    K_combined,
    x=labels,
    y=labels,
    title="Kronecker product: age ⊗ location mixing",
    color_continuous_scale="Greys",
)
fig.show()
print("combined[0,0] (young+urb ← young+urb) =", float(K_combined[0, 0]))


## Prem et al. contact matrices

Empirical age×location matrices for many countries are published with
[Prem et al. (2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005697)
and updated on [Kiesha Prem's GitHub](https://github.com/kieshaprem/synthetic-contact-matrices).
Loading and scaling those surveys is ledger WP9 — not part of this page. Once
loaded as an `ndarray`, pass them to `MixingMatrix` like any other matrix.


## Summary

| summer2 | summer4 |
|---|---|
| `strat.set_mixing_matrix` | `MixingMatrix` on `ForceOfInfection` |
| Time-varying matrix | `ArrayConst` blend with `piecewise(Time(), ...)` |
| Independent multi-axis matrices | Explicit `np.kron` (single FOI `group_by`) |
